In [1]:
import pandas as pd
import numpy as np
import nltk
import re
import os
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from collections import Counter

# Download NLTK data
nltk.download('vader_lexicon')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

PROCESSED_DIR = 'data/processed'
LMD_PATH      = 'data/Loughran-McDonald_MasterDictionary_1993-2024.csv'

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/xnazhang/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt to /Users/xnazhang/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/xnazhang/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/xnazhang/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
# ── 1. Load data ─────────────────────────────────────────────────────────────

df_transcripts = pd.read_csv(os.path.join(PROCESSED_DIR, 'transcripts_clean.csv'),
                             parse_dates=['date'])
df_returns     = pd.read_csv(os.path.join(PROCESSED_DIR, 'price_returns.csv'),
                             parse_dates=['date'])

print(f'Transcripts: {df_transcripts.shape}')
print(f'Returns:     {df_returns.shape}')
print(df_transcripts[['ticker', 'quarter', 'date', 'char_count']].head(8))

Transcripts: (16, 9)
Returns:     (16, 9)
  ticker quarter       date  char_count
0   AAPL      Q1 2022-01-28       49053
1   AAPL      Q2 2022-04-29       50212
2   AAPL      Q3 2022-07-28       48381
3   AAPL      Q4 2022-10-27       49061
4   MSFT      Q1 2023-10-24       55743
5   MSFT      Q2 2024-01-31       57543
6   MSFT      Q3 2024-04-25       57007
7   MSFT      Q4 2024-07-30       54855


In [4]:
# ── 2. Load Loughran-McDonald lexicon ────────────────────────────────────────

def load_lmd_lexicon(path: str) -> tuple[set, set]:
    """
    Load Loughran-McDonald Master Dictionary and return sets of
    positive and negative financial words.

    Returns:
        (positive_words: set, negative_words: set) — all lowercase
    """
    lmd = pd.read_csv(path)
    lmd.columns = [c.strip().title() for c in lmd.columns]

    pos_words = set(lmd[lmd['Positive'] != 0]['Word'].str.lower())
    neg_words = set(lmd[lmd['Negative'] != 0]['Word'].str.lower())

    print(f'L-MD Positive words: {len(pos_words):,}')
    print(f'L-MD Negative words: {len(neg_words):,}')
    return pos_words, neg_words


LMD_POS, LMD_NEG = load_lmd_lexicon(LMD_PATH)

L-MD Positive words: 354
L-MD Negative words: 2,355


In [5]:
# ── 3. L-MD sentiment scoring ────────────────────────────────────────────────

def lmd_sentiment(text: str, pos_words: set, neg_words: set) -> dict:
    """
    Score text using the Loughran-McDonald financial sentiment lexicon.

    Args:
        text:      Cleaned transcript text
        pos_words: Set of L-MD positive words
        neg_words: Set of L-MD negative words

    Returns:
        dict with lmd_pos_count, lmd_neg_count, lmd_total_words,
        lmd_pos_pct, lmd_neg_pct, lmd_net_score, lmd_top_pos, lmd_top_neg
    """
    tokens = re.findall(r'\b[a-z]+\b', text.lower())
    total  = len(tokens)

    if total == 0:
        return {k: 0 for k in ['lmd_pos_count', 'lmd_neg_count', 'lmd_total_words',
                                'lmd_pos_pct', 'lmd_neg_pct', 'lmd_net_score',
                                'lmd_top_pos', 'lmd_top_neg']}

    pos_hits = [t for t in tokens if t in pos_words]
    neg_hits = [t for t in tokens if t in neg_words]

    pos_count = len(pos_hits)
    neg_count = len(neg_hits)

    return {
        'lmd_pos_count':   pos_count,
        'lmd_neg_count':   neg_count,
        'lmd_total_words': total,
        'lmd_pos_pct':     round(pos_count / total, 6),
        'lmd_neg_pct':     round(neg_count / total, 6),
        'lmd_net_score':   round((pos_count - neg_count) / total, 6),
        'lmd_top_pos':     ', '.join([w for w, _ in Counter(pos_hits).most_common(5)]),
        'lmd_top_neg':     ', '.join([w for w, _ in Counter(neg_hits).most_common(5)]),
    }

In [6]:
# ── 4. VADER sentiment scoring ───────────────────────────────────────────────

sid = SentimentIntensityAnalyzer()


def vader_sentiment(text: str, chunk_size: int = 512) -> dict:
    """
    Score text using VADER, chunked for long documents.

    Splits transcript into chunks, scores each, and averages compound scores.

    Args:
        text:       Cleaned transcript text
        chunk_size: Word count per chunk

    Returns:
        dict with vader_compound, vader_pos, vader_neg, vader_neu
    """
    words  = text.split()
    chunks = [' '.join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]
    scores = [sid.polarity_scores(chunk) for chunk in chunks]

    return {
        'vader_compound': round(np.mean([s['compound'] for s in scores]), 6),
        'vader_pos':      round(np.mean([s['pos']      for s in scores]), 6),
        'vader_neg':      round(np.mean([s['neg']      for s in scores]), 6),
        'vader_neu':      round(np.mean([s['neu']      for s in scores]), 6),
    }

In [8]:
# ── 5. Score all transcripts ─────────────────────────────────────────────────

lmd_results   = []
vader_results = []
prep_results  = []

for _, row in df_transcripts.iterrows():
    full_text = str(row['full_text_clean'])
    prep_text = str(row['prepared_remarks_clean']) \
                if pd.notna(row['prepared_remarks_clean']) \
                and len(str(row['prepared_remarks_clean'])) > 100 \
                else full_text

    # Full transcript scores
    lmd_scores   = lmd_sentiment(full_text, LMD_POS, LMD_NEG)
    vader_scores = vader_sentiment(full_text)

    # Prepared remarks only (L-MD)
    prep_scores  = lmd_sentiment(prep_text, LMD_POS, LMD_NEG)

    lmd_results.append(lmd_scores)
    vader_results.append(vader_scores)
    prep_results.append({
        'prep_lmd_net_score': prep_scores['lmd_net_score'],
        'prep_lmd_pos_pct':   prep_scores['lmd_pos_pct'],
        'prep_lmd_neg_pct':   prep_scores['lmd_neg_pct'],
    })

    print(f"{row['ticker']} {row['quarter']} — "
          f"L-MD net: {lmd_scores['lmd_net_score']:+.4f} | "
          f"VADER compound: {vader_scores['vader_compound']:+.4f}")

AAPL Q1 — L-MD net: +0.0090 | VADER compound: +0.9926
AAPL Q2 — L-MD net: +0.0055 | VADER compound: +0.9821
AAPL Q3 — L-MD net: +0.0046 | VADER compound: +0.9713
AAPL Q4 — L-MD net: +0.0114 | VADER compound: +0.9221
MSFT Q1 — L-MD net: +0.0116 | VADER compound: +0.9330
MSFT Q2 — L-MD net: +0.0080 | VADER compound: +0.9868
MSFT Q3 — L-MD net: +0.0084 | VADER compound: +0.9828
MSFT Q4 — L-MD net: +0.0051 | VADER compound: +0.9903
NVDA Q1 — L-MD net: +0.0108 | VADER compound: +0.9858
NVDA Q2 — L-MD net: +0.0047 | VADER compound: +0.9776
NVDA Q3 — L-MD net: +0.0065 | VADER compound: +0.9774
NVDA Q4 — L-MD net: +0.0073 | VADER compound: +0.9775
JPM Q1 — L-MD net: -0.0009 | VADER compound: +0.9702
JPM Q2 — L-MD net: +0.0014 | VADER compound: +0.9716
JPM Q3 — L-MD net: -0.0025 | VADER compound: +0.9909
JPM Q4 — L-MD net: +0.0018 | VADER compound: +0.9518


In [9]:
# ── 6. Assemble NLP dataframe ────────────────────────────────────────────────

df_nlp = pd.concat([
    df_transcripts[['ticker', 'date', 'quarter', 'sector']].reset_index(drop=True),
    pd.DataFrame(lmd_results),
    pd.DataFrame(vader_results),
    pd.DataFrame(prep_results),
], axis=1)

print(f'NLP scores shape: {df_nlp.shape}')
print(df_nlp[['ticker', 'quarter', 'lmd_net_score', 'vader_compound']].to_string(index=False))

NLP scores shape: (16, 19)
ticker quarter  lmd_net_score  vader_compound
  AAPL      Q1       0.009034        0.992606
  AAPL      Q2       0.005526        0.982088
  AAPL      Q3       0.004640        0.971344
  AAPL      Q4       0.011423        0.922135
  MSFT      Q1       0.011648        0.932975
  MSFT      Q2       0.008039        0.986768
  MSFT      Q3       0.008446        0.982800
  MSFT      Q4       0.005058        0.990283
  NVDA      Q1       0.010825        0.985800
  NVDA      Q2       0.004667        0.977567
  NVDA      Q3       0.006476        0.977359
  NVDA      Q4       0.007337        0.977475
   JPM      Q1      -0.000919        0.970182
   JPM      Q2       0.001424        0.971645
   JPM      Q3      -0.002481        0.990886
   JPM      Q4       0.001834        0.951756


In [11]:
# ── 7. Merge with price returns ──────────────────────────────────────────────

df_merged = pd.merge(
    df_nlp,
    df_returns[['ticker', 'date', 'fiscal_year', 'return_1d', 'return_3d', 'return_5d', 'beat_miss']],
    on=['ticker', 'date'],
    how='inner'
)

print(f'\nMerged dataset shape: {df_merged.shape}')
print(f'Rows matched: {len(df_merged)} / {len(df_nlp)}')


Merged dataset shape: (16, 24)
Rows matched: 16 / 16


In [12]:
# ── 8. Correlation analysis ──────────────────────────────────────────────────

sentiment_cols = ['lmd_net_score', 'lmd_pos_pct', 'lmd_neg_pct',
                  'vader_compound', 'prep_lmd_net_score']
return_cols    = ['return_1d', 'return_3d', 'return_5d']

corr_matrix = df_merged[sentiment_cols + return_cols].corr()

print('\n── Sentiment vs. Return Correlations ──────────────────────────')
print(corr_matrix.loc[sentiment_cols, return_cols].round(3).to_string())

print('\n── L-MD vs VADER correlation ───────────────────────────────────')
print(f"Pearson r(lmd_net, vader_compound): "
      f"{df_merged['lmd_net_score'].corr(df_merged['vader_compound']):.3f}")


── Sentiment vs. Return Correlations ──────────────────────────
                    return_1d  return_3d  return_5d
lmd_net_score           0.157     -0.043     -0.063
lmd_pos_pct             0.497      0.347      0.189
lmd_neg_pct             0.496      0.615      0.405
vader_compound         -0.488     -0.208     -0.038
prep_lmd_net_score     -0.448     -0.512     -0.497

── L-MD vs VADER correlation ───────────────────────────────────
Pearson r(lmd_net, vader_compound): -0.296


In [13]:
# ── 9. Save ──────────────────────────────────────────────────────────────────

out_path = os.path.join(PROCESSED_DIR, 'merged_final.csv')
df_merged.to_csv(out_path, index=False)

print(f'\nFinal dataset saved → {out_path}')
print(f'Shape: {df_merged.shape}')
print(df_merged[['ticker', 'quarter', 'fiscal_year', 'lmd_net_score',
                 'vader_compound', 'return_1d', 'beat_miss']].to_string(index=False))


Final dataset saved → data/processed/merged_final.csv
Shape: (16, 24)
ticker quarter  fiscal_year  lmd_net_score  vader_compound  return_1d beat_miss
  AAPL      Q1         2022       0.009034        0.992606   0.026126      Beat
  AAPL      Q2         2022       0.005526        0.982088   0.001966      Beat
  AAPL      Q3         2022       0.004640        0.971344   0.032793      Beat
  AAPL      Q4         2022       0.011423        0.922135   0.075552      Beat
  MSFT      Q1         2024       0.011648        0.932975   0.030678      Beat
  MSFT      Q2         2024       0.008039        0.986768   0.015594      Beat
  MSFT      Q3         2024       0.008446        0.982800   0.018244      Beat
  MSFT      Q4         2024       0.005058        0.990283  -0.010806      Miss
  NVDA      Q1         2025       0.010825        0.985800  -0.037666      Miss
  NVDA      Q2         2025       0.004667        0.977567  -0.063848      Miss
  NVDA      Q3         2025       0.006476       